<a href="https://colab.research.google.com/github/Siddesh-2004/FindYourPhone/blob/main/notebook/featureBasedRecommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Imports**

In [53]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances  # ← add this

In [54]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Load & Clean
# ═══════════════════════════════════════════════════════════════
df = pd.read_csv("/content/phones.csv")

# Drop duplicates
df.drop_duplicates(inplace=True)

# Drop dead columns
dead_cols = ["img", "display pixels", "display frequency (in Hz)",
             "Punch Hole", "extended_upto", "core", "os_version"]
df.drop(columns=dead_cols, inplace=True)

# Extract numeric fields
df["ram"]            = df["ram"].str.extract(r"(\d+)").astype(float).astype("Int64")
df["storage"]        = df["ram (inbuilt)"].str.extract(r"(\d+)").astype(float).astype("Int64")
df["battery"]        = df["battery (in mAh)"].str.extract(r"(\d+)").astype(float).astype("Int64")
df["display_size"]   = df["display size"].str.extract(r"([\d.]+)").astype(float)
df["fast_charging"]  = df["fast charging"].str.extract(r"(\d+)").astype(float).astype("Int64")
df["frequency"]      = df["frequency"].str.extract(r"([\d.]+)").astype(float)
df["front_camera_mp"]= df["front_camera"].str.extract(r"(\d+)").astype(float).astype("Int64")
df["rear_camera_mp"] = df["rear_camera"].str.extract(r"(\d+)").astype(float).astype("Int64")

df.drop(columns=["ram (inbuilt)", "battery (in mAh)", "display size",
                  "fast charging", "front_camera", "rear_camera"], inplace=True)

# Fix booleans
for col in ["4G", "5G", "NFC", "VoLTE"]:
    df[col] = df[col].map(lambda x: 1 if str(x).strip().lower() == "true" else 0).astype("Int64")

# Clean os_brand
valid_os = {"Android", "iOS", "HarmonyOS"}
df["os_brand"] = df["os_brand"].apply(lambda x: x if x in valid_os else None)

print("After cleaning:", df.shape)



After cleaning: (814, 21)


In [55]:
features = ['ram', 'storage', 'battery', 'fast_charging',
            'front_camera_mp', 'rear_camera_mp', 'specs_score']

# Cap outliers via IQR
for col in features:
    df[col] = df[col].astype(float)  # Int64 doesn't support float clip bounds
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    df[col] = df[col].clip(lower=q1 - 1.5 * iqr, upper=q3 + 1.5 * iqr)

# Drop low-variance columns confirmed from variance check
df.drop(columns=["display_size", "rating"], inplace=True, errors="ignore")

print("After variance step:", df.shape)


After variance step: (814, 19)


In [56]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — Normalize Features
# ═══════════════════════════════════════════════════════════════
df_model = df.dropna(subset=features).reset_index(drop=True)

scaler = MinMaxScaler()
df_model[features] = scaler.fit_transform(df_model[features])

print("After normalization:", df_model.shape)



After normalization: (624, 19)


In [57]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — Rule-Based Filter
# ═══════════════════════════════════════════════════════════════
def rule_based_filter(df, budget=None, os_brand=None, need_5g=False, need_nfc=False):
    filtered = df.copy()
    if budget is not None:
        filtered = filtered[filtered["price"] <= budget]
    if os_brand is not None:
        filtered = filtered[filtered["os_brand"] == os_brand]
    if need_5g:
        filtered = filtered[filtered["5G"] == 1]
    if need_nfc:
        filtered = filtered[filtered["NFC"] == 1]
    return filtered.reset_index(drop=True)


In [58]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — Cosine Similarity Scorer and euclidean distance
# ═══════════════════════════════════════════════════════════════

def map_weights(feature_importance: dict) -> np.ndarray:
    """
    Converts user slider values (1–5) to weight multipliers (1.0–3.0).
    Formula: weight = 1.0 + (slider - 1) * 0.5
    Slider 1 → 1.0, Slider 3 → 2.0, Slider 5 → 3.0
    """
    return np.array([
        1.0 + (feature_importance.get(f, 3) - 1) * 0.5
        for f in features
    ])


def build_user_vector(ram=8, storage=128, battery=4500, fast_charging=33,
                      front_camera_mp=16, rear_camera_mp=50, specs_score=60):
    raw = pd.DataFrame([[ram, storage, battery, fast_charging,
                         front_camera_mp, rear_camera_mp, specs_score]],
                       columns=features)
    return scaler.transform(raw)


def euclidean_similarity(user_vector, phone_matrix, weights):
    """
    Computes normalized Euclidean similarity between user vector and all phones.

    Steps:
      1. Apply weights to both sides
      2. Compute row-wise Euclidean distance
      3. Normalize by theoretical max distance in weighted [0,1] space
         → max possible distance = sqrt(sum of weights²)
      4. Flip to similarity: 1 - normalized_distance

    Returns array of shape (n,) with values in [0, 1].
    """
    weighted_user   = user_vector * weights           # shape (1, 7)
    weighted_matrix = phone_matrix * weights          # shape (n, 7)

    # Row-wise Euclidean distance: shape (n,)
    diff      = weighted_matrix - weighted_user       # broadcasting
    distances = np.linalg.norm(diff, axis=1)

    # Theoretical max: farthest two points in weighted [0,1]^7 space
    max_distance = np.sqrt(np.sum(weights ** 2))

    # Normalize and flip to similarity
    return 1 - (distances / max_distance)


def score_phones(candidate_df, user_vector, weights):
    """
    Hybrid score = 0.6 × cosine_similarity + 0.4 × euclidean_similarity
    Both terms are in [0, 1], higher = better match.
    """
    weighted_user   = user_vector * weights
    weighted_matrix = candidate_df[features].values * weights

    cosine_scores    = cosine_similarity(weighted_user, weighted_matrix)[0]
    euclidean_scores = euclidean_similarity(user_vector, candidate_df[features].values, weights)

    hybrid_scores = 0.6 * cosine_scores + 0.4 * euclidean_scores

    result = candidate_df.copy()
    result["match_score"] = (hybrid_scores * 100).round(2)
    return result.sort_values("match_score", ascending=False).reset_index(drop=True)

In [59]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — recommend() function
# ═══════════════════════════════════════════════════════════════

DEFAULT_IMPORTANCE = {
    "ram": 3, "storage": 3, "battery": 3,
    "fast_charging": 3, "front_camera_mp": 3,
    "rear_camera_mp": 3, "specs_score": 3
}

def recommend(budget=None, os_brand=None, need_5g=False, need_nfc=False,
              ram=8, storage=128, battery=4500, fast_charging=33,
              front_camera_mp=16, rear_camera_mp=50, specs_score=60,
              feature_importance=None, top_n=10):

    # Fall back to neutral defaults if no importance provided
    importance = feature_importance if feature_importance is not None else DEFAULT_IMPORTANCE

    candidates = rule_based_filter(df_model, budget=budget, os_brand=os_brand,
                                   need_5g=need_5g, need_nfc=need_nfc)
    if candidates.empty:
        print("No phones match your filters.")
        return pd.DataFrame()

    user_vector = build_user_vector(ram=ram, storage=storage, battery=battery,
                                    fast_charging=fast_charging,
                                    front_camera_mp=front_camera_mp,
                                    rear_camera_mp=rear_camera_mp,
                                    specs_score=specs_score)

    weights = map_weights(importance)
    ranked  = score_phones(candidates, user_vector, weights)
    return ranked[["name", "price", "match_score"]].head(top_n)

In [60]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — 3 Persona Demos
# ═══════════════════════════════════════════════════════════════

personas = [
    {
        "name": "Student",
        "params": dict(
            budget=12000, os_brand="Android", need_5g=False,
            ram=4, storage=64, battery=5000, fast_charging=18,
            front_camera_mp=8, rear_camera_mp=13, specs_score=40,
            feature_importance={
                "battery": 5,       # longest battery life matters most
                "storage": 4,       # needs space for apps/files
                "ram": 3,
                "fast_charging": 2,
                "front_camera_mp": 2,
                "rear_camera_mp": 1,
                "specs_score": 3
            }
        )
    },
    {
        "name": "Photographer",
        "params": dict(
            budget=50000, os_brand="Android", need_5g=False,
            ram=8, storage=256, battery=4500, fast_charging=33,
            front_camera_mp=32, rear_camera_mp=108, specs_score=70,
            feature_importance={
                "rear_camera_mp": 5,   # primary camera is everything
                "front_camera_mp": 4,  # selfies/video calls matter too
                "storage": 4,          # raw photos eat storage
                "specs_score": 3,
                "ram": 2,
                "battery": 2,
                "fast_charging": 1
            }
        )
    },
    {
        "name": "Gamer",
        "params": dict(
            budget=40000, os_brand="Android", need_5g=True,
            ram=12, storage=256, battery=5000, fast_charging=65,
            front_camera_mp=16, rear_camera_mp=50, specs_score=80,
            feature_importance={
                "ram": 5,              # performance headroom
                "fast_charging": 5,    # back in game fast
                "battery": 4,          # long sessions
                "storage": 4,          # game installs are huge
                "specs_score": 4,
                "rear_camera_mp": 1,
                "front_camera_mp": 1
            }
        )
    },
]

for p in personas:
    print(f"\n=== {p['name']} ===")
    print(recommend(**p["params"]).to_string(index=False))


=== Student ===
                          name  price  match_score
                      Vivo Y02   8999        87.54
                   Realme C30s   6499        83.90
           itel Vision 3 Turbo   7100        83.88
itel Vision 3 (2GB RAM + 32GB)   5849        83.04
  Realme C30s (4GB RAM + 64GB)   7999        83.04
                      Poco C50   5749        82.93
          Xiaomi Redmi A1 Plus   7299        81.93
                     Vivo Y01A   8145        81.93
                     Vivo Y02s   8990        81.81
        Realme Narzo 50i Prime   6999        81.81

=== Photographer ===
                                   name  price  match_score
                               Vivo V30  33990        88.90
                               Vivo V28  28990        88.83
Samsung Galaxy A73 5G (8GB RAM + 256GB)  44999        87.53
                           Vivo V29 Pro  43990        87.04
                     Motorola Edge Plus  49999        85.77
                           Doogee V Max 

In [61]:
# ═══════════════════════════════════════════════════════════════
# CELL 9 — Golden Set Evaluation
# ═══════════════════════════════════════════════════════════════

def run_golden_set():
    passed = 0
    failed = 0

    def check(label, condition):
        nonlocal passed, failed
        if condition:
            print(f"  [ PASS ] {label}")
            passed += 1
        else:
            print(f"  [ FAIL ] {label}")
            failed += 1

    # ── helper: get full result rows (not just name/price/match_score) ──
    def recommend_full(budget=None, os_brand=None, need_5g=False, need_nfc=False,
                       ram=8, storage=128, battery=4500, fast_charging=33,
                       front_camera_mp=16, rear_camera_mp=50, specs_score=60,
                       feature_importance=None, top_n=10):
        importance = feature_importance if feature_importance is not None else DEFAULT_IMPORTANCE
        candidates = rule_based_filter(df_model, budget=budget, os_brand=os_brand,
                                       need_5g=need_5g, need_nfc=need_nfc)
        if candidates.empty:
            return pd.DataFrame()
        user_vector = build_user_vector(ram=ram, storage=storage, battery=battery,
                                        fast_charging=fast_charging,
                                        front_camera_mp=front_camera_mp,
                                        rear_camera_mp=rear_camera_mp,
                                        specs_score=specs_score)
        weights = map_weights(importance)
        ranked  = score_phones(candidates, user_vector, weights)
        return ranked.head(top_n)   # full df, all columns

    dataset_avg_battery = df_model["battery"].mean()  # normalized [0,1]

    # ───────────────────────────────────────────────
    # TEST GROUP 1 — Student
    # ───────────────────────────────────────────────
    print("\n=== Student ===")
    student_importance = {
        "battery": 5, "storage": 4, "ram": 3,
        "fast_charging": 2, "front_camera_mp": 2,
        "rear_camera_mp": 1, "specs_score": 3
    }
    student_results = recommend_full(
        budget=12000, os_brand="Android", need_5g=False,
        ram=4, storage=64, battery=5000, fast_charging=18,
        front_camera_mp=8, rear_camera_mp=13, specs_score=40,
        feature_importance=student_importance
    )

    check(
        "All top 5 results are within budget (≤ ₹12,000)",
        (student_results.head(5)["price"] <= 12000).all()
    )
    check(
        "Top 1 result is within budget",
        student_results.iloc[0]["price"] <= 12000
    )
    check(
        "Top 5 average battery is above dataset average",
        student_results.head(5)["battery"].mean() > dataset_avg_battery
    )
    check(
        "Top 1 match score is above 80",
        student_results.iloc[0]["match_score"] > 80
    )

    # ───────────────────────────────────────────────
    # TEST GROUP 2 — Photographer
    # ───────────────────────────────────────────────
    print("\n=== Photographer ===")
    photographer_importance = {
        "rear_camera_mp": 5, "front_camera_mp": 4, "storage": 4,
        "specs_score": 3, "ram": 2, "battery": 2, "fast_charging": 1
    }
    photographer_results = recommend_full(
        budget=50000, os_brand="Android", need_5g=False,
        ram=8, storage=256, battery=4500, fast_charging=33,
        front_camera_mp=32, rear_camera_mp=108, specs_score=70,
        feature_importance=photographer_importance
    )

    check(
        "All top 5 results are within budget (≤ ₹50,000)",
        (photographer_results.head(5)["price"] <= 50000).all()
    )
    check(
        "Top 5 rear_camera_mp is above dataset average",
        photographer_results.head(5)["rear_camera_mp"].mean() > df_model["rear_camera_mp"].mean()
    )
    check(
        "Top 5 front_camera_mp is above dataset average",
        photographer_results.head(5)["front_camera_mp"].mean() > df_model["front_camera_mp"].mean()
    )
    check(
        "Top 1 match score is above 85",
        photographer_results.iloc[0]["match_score"] > 85
    )

    # ───────────────────────────────────────────────
    # TEST GROUP 3 — Gamer
    # ───────────────────────────────────────────────
    print("\n=== Gamer ===")
    gamer_importance = {
        "ram": 5, "fast_charging": 5, "battery": 4,
        "storage": 4, "specs_score": 4,
        "rear_camera_mp": 1, "front_camera_mp": 1
    }
    gamer_results = recommend_full(
        budget=40000, os_brand="Android", need_5g=True,
        ram=12, storage=256, battery=5000, fast_charging=65,
        front_camera_mp=16, rear_camera_mp=50, specs_score=80,
        feature_importance=gamer_importance
    )

    check(
        "All top 5 results have 5G",
        (gamer_results.head(5)["5G"] == 1).all()
    )
    check(
        "All top 5 results are within budget (≤ ₹40,000)",
        (gamer_results.head(5)["price"] <= 40000).all()
    )
    check(
        "Top 5 average ram is above dataset average",
        gamer_results.head(5)["ram"].mean() > df_model["ram"].mean()
    )
    check(
        "Top 5 average fast_charging is above dataset average",
        gamer_results.head(5)["fast_charging"].mean() > df_model["fast_charging"].mean()
    )
    check(
        "Top 1 match score is above 90",
        gamer_results.iloc[0]["match_score"] > 90
    )

    # ───────────────────────────────────────────────
    # TEST GROUP 4 — Edge Cases
    # ───────────────────────────────────────────────
    print("\n=== Edge Cases ===")

    # Impossible budget — should return empty
    edge_results = recommend_full(budget=100, os_brand="Android")
    check(
        "Impossible budget (₹100) returns empty results",
        edge_results.empty
    )

    # Extreme high weights — should not crash and still return results
    try:
        extreme_results = recommend_full(
            budget=50000,
            feature_importance={f: 5 for f in features}
        )
        check(
            "Extreme weights (all sliders at 5) still returns results without crash",
            not extreme_results.empty
        )
    except Exception as e:
        check(f"Extreme weights (all sliders at 5) still returns results without crash — ERROR: {e}", False)

    # ───────────────────────────────────────────────
    # TEST GROUP 5 — Score Sanity
    # ───────────────────────────────────────────────
    print("\n=== Score Sanity ===")

    # A user who weights rear_camera heavily should score the same phone
    # higher than a user who doesn't care about camera at all
    base_params = dict(
        budget=50000, os_brand="Android",
        ram=8, storage=256, battery=4500,
        fast_charging=33, front_camera_mp=32,
        rear_camera_mp=108, specs_score=70
        # top_n removed — passed explicitly per call below
    )

    high_camera = recommend_full(**base_params, top_n=1, feature_importance={
        "rear_camera_mp": 5, "front_camera_mp": 4, "storage": 3,
        "ram": 3, "battery": 3, "fast_charging": 3, "specs_score": 3
    })

    # Get score of the same top phone under both weight configs
    top_phone_name = high_camera.iloc[0]["name"]
    high_score = high_camera.iloc[0]["match_score"]

    low_camera_full = recommend_full(**base_params, top_n=50, feature_importance={
        "rear_camera_mp": 1, "front_camera_mp": 1, "storage": 3,
        "ram": 3, "battery": 3, "fast_charging": 3, "specs_score": 3
    })
    low_score_row = low_camera_full[low_camera_full["name"] == top_phone_name]
    low_score = low_score_row.iloc[0]["match_score"] if not low_score_row.empty else 0

    check(
        f"'{top_phone_name}' scores higher when camera weight=5 vs weight=1",
        high_score > low_score
    )

    # ───────────────────────────────────────────────
    # TEST GROUP 6 — Filter Correctness
    # ───────────────────────────────────────────────
    print("\n=== Filter Correctness ===")

    nfc_results = recommend_full(budget=30000, need_nfc=True, top_n=10)
    check(
        "need_nfc=True — all top 10 results have NFC",
        (nfc_results["NFC"] == 1).all()
    )

    ios_results = recommend_full(budget=150000, os_brand="iOS", top_n=10)
    check(
        "os_brand='iOS' — all top 10 results are iOS",
        (ios_results["os_brand"] == "iOS").all()
    )

    # ───────────────────────────────────────────────
    # TEST GROUP 7 — Cross-Persona Comparison
    # ───────────────────────────────────────────────
    print("\n=== Cross-Persona Comparison ===")

    # Both personas, same budget, no OS/connectivity filters
    # Photographer should surface phones with higher avg rear_camera_mp than Gamer
    photographer_cross = recommend_full(
        budget=40000, top_n=5,
        ram=8, storage=256, battery=4500, fast_charging=33,
        front_camera_mp=32, rear_camera_mp=108, specs_score=70,
        feature_importance={
            "rear_camera_mp": 5, "front_camera_mp": 4, "storage": 4,
            "specs_score": 3, "ram": 2, "battery": 2, "fast_charging": 1
        }
    )
    gamer_cross = recommend_full(
        budget=40000, top_n=5,
        ram=12, storage=256, battery=5000, fast_charging=65,
        front_camera_mp=16, rear_camera_mp=50, specs_score=80,
        feature_importance={
            "ram": 5, "fast_charging": 5, "battery": 4,
            "storage": 4, "specs_score": 4,
            "rear_camera_mp": 1, "front_camera_mp": 1
        }
    )

    photographer_avg_camera = photographer_cross["rear_camera_mp"].mean()
    gamer_avg_camera        = gamer_cross["rear_camera_mp"].mean()

    check(
        "Photographer top 5 has higher avg rear_camera_mp than Gamer top 5",
        photographer_avg_camera > gamer_avg_camera
    )

    # ───────────────────────────────────────────────
    # SUMMARY
    # ───────────────────────────────────────────────
    total = passed + failed
    print(f"\n{'═'*45}")
    print(f"  Results: {passed}/{total} passed", end="  ")
    print("✓ All good!" if failed == 0 else f"✗ {failed} assertion(s) failed — check above")
    print(f"{'═'*45}")

run_golden_set()


=== Student ===
  [ PASS ] All top 5 results are within budget (≤ ₹12,000)
  [ PASS ] Top 1 result is within budget
  [ PASS ] Top 5 average battery is above dataset average
  [ PASS ] Top 1 match score is above 80

=== Photographer ===
  [ PASS ] All top 5 results are within budget (≤ ₹50,000)
  [ PASS ] Top 5 rear_camera_mp is above dataset average
  [ PASS ] Top 5 front_camera_mp is above dataset average
  [ PASS ] Top 1 match score is above 85

=== Gamer ===
  [ PASS ] All top 5 results have 5G
  [ PASS ] All top 5 results are within budget (≤ ₹40,000)
  [ PASS ] Top 5 average ram is above dataset average
  [ PASS ] Top 5 average fast_charging is above dataset average
  [ PASS ] Top 1 match score is above 90

=== Edge Cases ===
  [ PASS ] Impossible budget (₹100) returns empty results
  [ PASS ] Extreme weights (all sliders at 5) still returns results without crash

=== Score Sanity ===
  [ PASS ] 'Samsung Galaxy A73 5G (8GB RAM + 256GB)' scores higher when camera weight=5 vs weig